# 76. The VRP with Split Deliveries (SDVRP)
## Tier 6: The Autonomous & Self-Optimizing Ecosystem

### Key assumptions
- Vehicles are autonomous agents with decision-making capabilities
- Central auctioneer coordinates task allocation through bidding
- Agents negotiate and collaborate to optimize global objectives
- System continuously self-optimizes through emergent behavior

### Approach (step-by-step)
1. **Agent Architecture**: Define vehicle agents with capabilities and preferences
2. **Task Broadcasting**: Decompose deliveries into atomic tasks
3. **Auction Mechanism**: Competitive bidding for task allocation
4. **Contract Net Protocol**: Formal agreements between agents
5. **Dynamic Re-bidding**: Continuous optimization as conditions change

### What to look for in the results
- Emergent routing patterns from agent interactions
- Auction efficiency and task allocation quality
- System performance under dynamic conditions
- Comparison with centrally planned approaches

### Concrete example (from the source)
We'll implement a multi-agent system for SDVRP:
- Same 4-customer instance with agent-based coordination
- 2 vehicle agents with bidding and negotiation capabilities
- Central auctioneer for task allocation
- Dynamic re-bidding for continuous optimization

In [1]:
# Import required packages for Multi-Agent System
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from math import sqrt, exp
import random
import time
from datetime import datetime, timedelta
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(76)
random.seed(76)

print("Libraries imported successfully for Multi-Agent System")

Libraries imported successfully for Multi-Agent System


Libraries imported successfully for Multi-Agent System


Libraries imported successfully for Multi-Agent System


Libraries imported successfully for Multi-Agent System


Libraries imported successfully for Multi-Agent System


In [ ]:
# Define problem data structures (same as previous tiers)
class SDVRPInstance:
    """Class to represent SDVRP problem instance"""
    def __init__(self, depot_coords, customer_coords, demands, vehicle_capacities):
        self.depot = depot_coords
        self.customers = customer_coords
        self.demands = demands
        self.capacities = vehicle_capacities
        self.n_customers = len(customer_coords)
        self.n_vehicles = len(vehicle_capacities)
        
    def calculate_distance_matrix(self):
        """Calculate Euclidean distance matrix between all nodes"""
        all_nodes = [self.depot] + self.customers
        n_nodes = len(all_nodes)
        distances = np.zeros((n_nodes, n_nodes))
        
        for i in range(n_nodes):
            for j in range(n_nodes):
                if i != j:
                    x1, y1 = all_nodes[i]
                    x2, y2 = all_nodes[j]
                    distances[i][j] = sqrt((x2 - x1)**2 + (y2 - y1)**2)
        
        return distances

# Create the same instance as previous tiers
depot_coords = (0, 0)
customer_coords = [(10, 15), (20, 5), (15, 20), (25, 10)]
demands = [70, 130, 60, 80]  # Customer demands
vehicle_capacities = [100, 100]  # Two vehicles with capacity 100

instance = SDVRPInstance(depot_coords, customer_coords, demands, vehicle_capacities)
distance_matrix = instance.calculate_distance_matrix()

print(f"SDVRP Instance for Multi-Agent System:")
print(f"- Depot: {depot_coords}")
print(f"- Customers: {len(customer_coords)} with demands: {demands}")
print(f"- Vehicles: {len(vehicle_capacities)} with capacities: {vehicle_capacities}")
print(f"- Total demand: {sum(demands)} units")
print(f"- Total capacity: {sum(vehicle_capacities)} units")

In [ ]:
# Task definition for agent coordination
class DeliveryTask:
    """Represents a delivery task that can be bid on by agents"""
    
    def __init__(self, task_id, customer_id, quantity, location, priority='normal'):
        self.task_id = task_id
        self.customer_id = customer_id
        self.quantity = quantity
        self.location = location
        self.priority = priority
        self.assigned_agent = None
        self.bid_history = []
        self.created_time = time.time()
        
    def add_bid(self, agent_id, bid_amount, confidence):
        """Add a bid from an agent"""
        self.bid_history.append({
            'agent_id': agent_id,
            'bid_amount': bid_amount,
            'confidence': confidence,
            'timestamp': time.time()
        })
    
    def get_lowest_bid(self):
        """Get the lowest bid amount"""
        if not self.bid_history:
            return None
        
        lowest_bid = min(self.bid_history, key=lambda x: x['bid_amount'])
        return lowest_bid
    
    def __str__(self):
        return f"Task-{self.task_id}: Customer {self.customer_id}, Qty {self.quantity}, Priority {self.priority}"

# Vehicle Agent class
class VehicleAgent:
    """Autonomous vehicle agent with bidding and routing capabilities"""
    
    def __init__(self, agent_id, capacity, initial_location, distance_matrix):
        self.agent_id = agent_id
        self.capacity = capacity
        self.current_location = initial_location
        self.distance_matrix = distance_matrix
        
        # Agent state
        self.current_load = 0
        self.assigned_tasks = []
        self.completed_tasks = []
        self.route = [initial_location]
        
        # Agent characteristics
        self.cost_per_km = 2.5  # Operating cost
        self.max_range = 500  # Maximum range in km
        self.current_fuel = 100.0  # Fuel percentage
        
        # Bidding strategy parameters
        self.risk_tolerance = 0.5  # 0 = risk-averse, 1 = risk-seeking
        self.competitiveness = 0.7  # How aggressively to bid
        self.workload_preference = 0.6  # Preference for balanced workload
        
        # Learning parameters
        self.bid_success_history = []
        self.profit_history = []
        
    def calculate_bid_cost(self, task):
        """Calculate the cost for the agent to perform a task"""
        # Distance from current location to task location
        if self.current_location == 0:  # At depot
            from_location = self.distance_matrix[0][task.customer_id + 1]
        else:
            from_location = self.distance_matrix[self.current_location][task.customer_id + 1]
        
        # Distance from task location back to depot (if needed)
        to_depot = self.distance_matrix[task.customer_id + 1][0]
        
        # Total distance for this task
        if self.current_load + task.quantity > self.capacity:
            # Need to return to depot first
            total_distance = (self.distance_matrix[self.current_location][0] + 
                            from_location + to_depot)
        else:
            total_distance = from_location
        
        # Base cost
        base_cost = total_distance * self.cost_per_km
        
        # Capacity utilization factor
        utilization = (self.current_load + task.quantity) / self.capacity
        utilization_factor = 1.0 - (utilization - 0.5) ** 2  # Prefer 50% utilization
        
        # Workload balance factor
        workload_factor = 1.0
        if len(self.assigned_tasks) > 3:
            workload_factor = 1.2  # Penalty for high workload
        elif len(self.assigned_tasks) == 0:
            workload_factor = 0.8  # Bonus for being idle
        
        # Risk adjustment
        risk_adjustment = 1.0 + self.risk_tolerance * 0.1
        
        # Competitiveness adjustment
        competitive_adjustment = 1.0 - self.competitiveness * 0.15
        
        # Final bid cost
        bid_cost = base_cost * utilization_factor * workload_factor * risk_adjustment * competitive_adjustment
        
        return bid_cost
    
    def calculate_bid_confidence(self, task):
        """Calculate confidence in successfully completing the task"""
        # Base confidence
        base_confidence = 0.8
        
        # Capacity constraint
        if self.current_load + task.quantity > self.capacity:
            capacity_confidence = 0.3
        else:
            capacity_confidence = 1.0
        
        # Fuel constraint
        distance_to_task = self.distance_matrix[self.current_location][task.customer_id + 1]
        fuel_needed = distance_to_task * 0.1  # 0.1% per km
        if self.current_fuel < fuel_needed:
            fuel_confidence = 0.2
        else:
            fuel_confidence = 1.0
        
        # Experience factor (based on completed tasks)
        experience_factor = min(1.0, 0.7 + len(self.completed_tasks) * 0.05)
        
        # Distance factor (prefer closer tasks)
        max_distance = np.max(self.distance_matrix)
        distance_factor = 1.0 - (distance_to_task / max_distance) * 0.2
        
        confidence = base_confidence * capacity_confidence * fuel_confidence * experience_factor * distance_factor
        return max(0.1, min(1.0, confidence))
    
    def place_bid(self, task):
        """Place a bid on a task"""
        bid_cost = self.calculate_bid_cost(task)
        confidence = self.calculate_bid_confidence(task)
        
        # Add some randomness to avoid predictable bidding
        noise = np.random.normal(0, 0.05)
        bid_cost *= (1 + noise)
        
        task.add_bid(self.agent_id, bid_cost, confidence)
        
        return bid_cost, confidence
    
    def assign_task(self, task):
        """Assign a task to this agent"""
        self.assigned_tasks.append(task)
        self.current_load += task.quantity
        task.assigned_agent = self.agent_id
        
        # Update route
        if self.current_location != task.customer_id + 1:
            self.route.append(task.customer_id + 1)
        
        self.current_location = task.customer_id + 1
        
        # Update fuel
        distance = self.distance_matrix[self.route[-2] if len(self.route) > 1 else 0][task.customer_id + 1]
        self.current_fuel -= distance * 0.1
    
    def complete_task(self, task):
        """Mark a task as completed"""
        if task in self.assigned_tasks:
            self.assigned_tasks.remove(task)
            self.completed_tasks.append(task)
            self.current_load -= task.quantity
            
            # Update location
            self.current_location = task.customer_id + 1
    
    def return_to_depot(self):
        """Return vehicle to depot"""
        if self.current_location != 0:
            self.route.append(0)
            self.current_location = 0
            
            # Update fuel for return trip
            distance = self.distance_matrix[self.route[-2]][0]
            self.current_fuel -= distance * 0.1
    
    def calculate_route_distance(self):
        """Calculate total distance of current route"""
        total_distance = 0
        for i in range(len(self.route) - 1):
            total_distance += self.distance_matrix[self.route[i]][self.route[i+1]]
        return total_distance
    
    def get_status(self):
        """Get current agent status"""
        return {
            'agent_id': self.agent_id,
            'current_location': self.current_location,
            'current_load': self.current_load,
            'capacity': self.capacity,
            'utilization': (self.current_load / self.capacity) * 100,
            'assigned_tasks': len(self.assigned_tasks),
            'completed_tasks': len(self.completed_tasks),
            'current_fuel': self.current_fuel,
            'route': self.route.copy()
            'route_distance': self.calculate_route_distance()
        }

print("Agent classes defined successfully")

In [2]:
# Central Auctioneer for task allocation
class CentralAuctioneer:
    """Central auctioneer that coordinates task allocation"""
    
    def __init__(self, instance, distance_matrix):
        self.instance = instance
        self.distance_matrix = distance_matrix
        self.agents = []
        self.tasks = []
        self.completed_tasks = []
        
        # Auction parameters
        self.min_bidding_rounds = 2
        self.max_bidding_rounds = 5
        self.bid_timeout = 5.0  # seconds
        
        # Performance tracking
        self.auction_history = []
        self.efficiency_metrics = []
        
    def register_agent(self, agent):
        """Register an agent with the auctioneer"""
        self.agents.append(agent)
        print(f"Agent {agent.agent_id} registered (Capacity: {agent.capacity})")
    
    def create_tasks(self, demands, granularity=10):
        """Create atomic tasks from customer demands"""
        self.tasks = []
        task_id = 0
        
        for customer_id, demand in enumerate(demands):
            # Split large demands into smaller tasks
            remaining_quantity = demand
            
            while remaining_quantity > 0:
                task_quantity = min(granularity, remaining_quantity)
                
                # Determine priority based on demand size
                if demand > 100:
                    priority = 'high'
                elif demand > 50:
                    priority = 'normal'
                else:
                    priority = 'low'
                
                task = DeliveryTask(
                    task_id=task_id,
                    customer_id=customer_id + 1,  # +1 because depot is 0
                    quantity=task_quantity,
                    location=instance.customers[customer_id],
                    priority=priority
                )
                
                self.tasks.append(task)
                remaining_quantity -= task_quantity
                task_id += 1
        
        print(f"Created {len(self.tasks)} tasks from {len(demands)} customer demands")
        return self.tasks
    
    def run_auction_round(self, task):
        """Run a single auction round for a task"""
        print(f"\n--- Auction Round for Task {task.task_id} ---")
        print(f"Task: {task}")
        
        # Clear previous bids
        task.bid_history = []
        
        # Collect bids from all agents
        bids = []
        for agent in self.agents:
            # Check if agent can handle the task
            if agent.current_load + task.quantity <= agent.capacity and agent.current_fuel > 10:
                bid_cost, confidence = agent.place_bid(task)
                bids.append({
                    'agent_id': agent.agent_id,
                    'bid_cost': bid_cost,
                    'confidence': confidence
                })
                print(f"  Agent {agent.agent_id}: Bid = {bid_cost:.2f}, Confidence = {confidence:.2f}")
            else:
                print(f"  Agent {agent.agent_id}: Cannot bid (capacity/fuel constraint)")
        
        # Award task to lowest bid
        if bids:
            winning_bid = min(bids, key=lambda x: x['bid_cost'])
            winning_agent = next(agent for agent in self.agents if agent.agent_id == winning_bid['agent_id'])
            
            # Assign task
            winning_agent.assign_task(task)
            
            print(f"  Winner: Agent {winning_bid['agent_id']} with bid {winning_bid['bid_cost']:.2f}")
            
            # Record auction result
            self.auction_history.append({
                'task_id': task.task_id,
                'winning_agent': winning_bid['agent_id'],
                'winning_bid': winning_bid['bid_cost'],
                'num_bids': len(bids),
                'timestamp': time.time()
            })
            
            return winning_agent
        else:
            print(f"  No bids received - task remains unassigned")
            return None
    
    def run_complete_auction(self):
        """Run complete auction for all tasks"""
        print(f"\n=== COMPLETE AUCTION ===")
        print(f"Agents: {len(self.agents)}, Tasks: {len(self.tasks)}")
        
        unassigned_tasks = self.tasks.copy()
        assigned_tasks = []
        
        auction_round = 1
        
        while unassigned_tasks and auction_round <= self.max_bidding_rounds:
            print(f"\n--- Auction Round {auction_round} ---")
            print(f"Unassigned tasks: {len(unassigned_tasks)}")
            
            tasks_to_remove = []
            
            for task in unassigned_tasks:
                winning_agent = self.run_auction_round(task)
                if winning_agent:
                    assigned_tasks.append(task)
                    tasks_to_remove.append(task)
            
            # Remove assigned tasks
            for task in tasks_to_remove:
                unassigned_tasks.remove(task)
            
            auction_round += 1
            
            # Check if minimum rounds reached
            if not unassigned_tasks and auction_round > self.min_bidding_rounds:
                break
        
        print(f"\n=== AUCTION RESULTS ===")
        print(f"Assigned tasks: {len(assigned_tasks)}")
        print(f"Unassigned tasks: {len(unassigned_tasks)}")
        print(f"Total auction rounds: {auction_round - 1}")
        
        return assigned_tasks, unassigned_tasks
    
    def calculate_system_metrics(self):
        """Calculate system-wide performance metrics"""
        total_distance = 0
        total_utilization = 0
        total_tasks_assigned = 0
        total_tasks_completed = 0
        
        for agent in self.agents:
            total_distance += agent.calculate_route_distance()
            total_utilization += (agent.current_load / agent.capacity) * 100
            total_tasks_assigned += len(agent.assigned_tasks)
            total_tasks_completed += len(agent.completed_tasks)
        
        avg_utilization = total_utilization / len(self.agents) if self.agents else 0
        
        metrics = {
            'total_distance': total_distance,
            'avg_utilization': avg_utilization,
            'total_tasks_assigned': total_tasks_assigned,
            'total_tasks_completed': total_tasks_completed,
            'num_agents': len(self.agents),
            'num_tasks': len(self.tasks),
            'assignment_rate': (total_tasks_assigned / len(self.tasks)) * 100 if self.tasks else 0
        }
        
        self.efficiency_metrics.append(metrics)
        return metrics

print("Central Auctioneer class defined")

Central Auctioneer class defined


Central Auctioneer class defined


Central Auctioneer class defined


Central Auctioneer class defined


Central Auctioneer class defined


In [ ]:
# Multi-Agent System Simulation
def run_multi_agent_simulation(instance, distance_matrix):
    """Run complete multi-agent simulation"""
    
    print("=== MULTI-AGENT SYSTEM SIMULATION ===")
    
    # Initialize auctioneer
    auctioneer = CentralAuctioneer(instance, distance_matrix)
    
    # Create and register agents
    for i in range(instance.n_vehicles):
        agent = VehicleAgent(
            agent_id=i,
            capacity=instance.capacities[i],
            initial_location=0,  # Start at depot
            distance_matrix=distance_matrix
        )
        auctioneer.register_agent(agent)
    
    # Create tasks from demands
    tasks = auctioneer.create_tasks(instance.demands, granularity=20)
    
    # Run auction
    assigned_tasks, unassigned_tasks = auctioneer.run_complete_auction()
    
    # Calculate final metrics
    final_metrics = auctioneer.calculate_system_metrics()
    
    # Get agent statuses
    agent_statuses = [agent.get_status() for agent in auctioneer.agents]
    
    print(f"\n=== FINAL SYSTEM STATE ===")
    print(f"Total distance: {final_metrics['total_distance']:.2f}")
    print(f"Average utilization: {final_metrics['avg_utilization']:.1f}%")
    print(f"Task assignment rate: {final_metrics['assignment_rate']:.1f}%")
    print(f"Total tasks assigned: {final_metrics['total_tasks_assigned']}")
    print(f"Total tasks completed: {final_metrics['total_tasks_completed']}")
    
    return auctioneer, agent_statuses, final_metrics

# Run the simulation
auctioneer, agent_statuses, final_metrics = run_multi_agent_simulation(instance, distance_matrix)

In [ ]:
# Dynamic Re-bidding Simulation
def simulate_dynamic_rebidding(auctioneer, instance, distance_matrix):
    """Simulate dynamic re-bidding as conditions change"""
    
    print("\n=== DYNAMIC RE-BIDDING SIMULATION ===")
    
    # Simulate multiple time periods with changing conditions
    time_periods = 3
    rebidding_history = []
    
    for period in range(time_periods):
        print(f"\n--- Time Period {period + 1} ---")
        
        # Simulate environmental changes
        if period == 1:
            # Traffic congestion increases
            print("Traffic congestion detected - adjusting agent costs")
            for agent in auctioneer.agents:
                agent.cost_per_km *= 1.3  # 30% cost increase
        elif period == 2:
            # Fuel prices change
            print("Fuel price increase - adjusting agent behavior")
            for agent in auctioneer.agents:
                agent.competitiveness *= 0.8  # Less competitive due to higher costs
        
        # Check for re-bidding opportunities
        rebidding_opportunities = 0
        
        for agent in auctioneer.agents:
            # Check if agent has high workload
            if len(agent.assigned_tasks) > 3:
                # Offer to trade tasks
                print(f"Agent {agent.agent_id} has high workload ({len(agent.assigned_tasks)} tasks)")
                
                # Try to trade with other agents
                for other_agent in auctioneer.agents:
                    if other_agent.agent_id != agent.agent_id and len(other_agent.assigned_tasks) < 2:
                        # Find a task to trade
                        if agent.assigned_tasks:
                            task_to_trade = agent.assigned_tasks[0]
                            
                            # Calculate trade benefit
                            current_cost = agent.calculate_bid_cost(task_to_trade)
                            other_cost = other_agent.calculate_bid_cost(task_to_trade)
                            
                            if other_cost < current_cost * 0.9:  # 10% improvement
                                print(f"  Trading task {task_to_trade.task_id} to Agent {other_agent.agent_id}")
                                print(f"  Cost improvement: {current_cost:.2f} → {other_cost:.2f}")
                                
                                # Execute trade
                                agent.assigned_tasks.remove(task_to_trade)
                                agent.current_load -= task_to_trade.quantity
                                
                                other_agent.assign_task(task_to_trade)
                                rebidding_opportunities += 1
                                break
        
        # Recalculate metrics
        period_metrics = auctioneer.calculate_system_metrics()
        rebidding_history.append({
            'period': period + 1,
            'metrics': period_metrics,
            'rebidding_opportunities': rebidding_opportunities
        })
        
        print(f"Period {period + 1} Metrics:")
        print(f"  Total distance: {period_metrics['total_distance']:.2f}")
        print(f"  Average utilization: {period_metrics['avg_utilization']:.1f}%")
        print(f"  Re-bidding opportunities: {rebidding_opportunities}")
    
    return rebidding_history

# Run dynamic re-bidding simulation
rebidding_history = simulate_dynamic_rebidding(auctioneer, instance, distance_matrix)

In [ ]:
# Visualize Multi-Agent System results
def visualize_multi_agent_results(agent_statuses, final_metrics, rebidding_history):
    """Comprehensive visualization of multi-agent system"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Agent routes and locations
    ax1.scatter(instance.depot[0], instance.depot[1], s=300, c='red', marker='s', 
               label='Depot', zorder=5, edgecolors='black', linewidth=2)
    
    # Plot customers
    for i, coord in enumerate(instance.customers):
        ax1.scatter(coord[0], coord[1], s=150, c='lightgray', marker='o', 
                   edgecolors='black', linewidth=2, zorder=4)
        ax1.annotate(f'C{i+1}\n({instance.demands[i]})', coord, 
                    xytext=(5, 5), textcoords='offset points', 
                    fontsize=10, fontweight='bold')
    
    # Plot agent routes
    colors = ['blue', 'green', 'red', 'purple', 'orange']
    
    for status in agent_statuses:
        agent_id = status['agent_id']
        route = status['route']
        
        if len(route) > 1:
            for i in range(len(route) - 1):
                start_node = route[i]
                end_node = route[i + 1]
                
                if start_node == 0:
                    start_coord = instance.depot
                else:
                    start_coord = instance.customers[start_node - 1]
                
                if end_node == 0:
                    end_coord = instance.depot
                else:
                    end_coord = instance.customers[end_node - 1]
                
                ax1.plot([start_coord[0], end_coord[0]], [start_coord[1], end_coord[1]], 
                        color=colors[agent_id], linewidth=2, alpha=0.7, 
                        label=f'Agent {agent_id+1}' if i == 0 else '')
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.set_title('Multi-Agent System: Agent Routes')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal')
    
    # Plot 2: Agent performance comparison
    agent_ids = [s['agent_id'] + 1 for s in agent_statuses]
    utilizations = [s['utilization'] for s in agent_statuses]
    distances = [s['route_distance'] for s in agent_statuses]
    tasks_assigned = [s['assigned_tasks'] for s in agent_statuses]
    
    x = np.arange(len(agent_ids))
    width = 0.2
    
    bars1 = ax2.bar(x - width, utilizations, width, label='Utilization (%)', 
                    color='blue', alpha=0.7)
    bars2 = ax2.bar(x, distances, width, label='Route Distance', 
                    color='green', alpha=0.7)
    bars3 = ax2.bar(x + width, tasks_assigned, width, label='Tasks Assigned', 
                    color='orange', alpha=0.7)
    
    ax2.set_xlabel('Agents')
    ax2.set_ylabel('Values')
    ax2.set_title('Agent Performance Comparison')
    ax2.set_xticks(x)
    ax2.set_xticklabels([f'Agent {i}' for i in agent_ids])
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Dynamic re-bidding performance
    if rebidding_history:
        periods = [h['period'] for h in rebidding_history]
        distances = [h['metrics']['total_distance'] for h in rebidding_history]
        utilizations = [h['metrics']['avg_utilization'] for h in rebidding_history]
        opportunities = [h['rebidding_opportunities'] for h in rebidding_history]
        
        ax3_twin = ax3.twinx()
        ax3.plot(periods, distances, 'b-', linewidth=2, marker='o', label='Total Distance')
        ax3_twin.plot(periods, utilizations, 'r-', linewidth=2, marker='s', label='Avg Utilization')
        
        # Add re-bidding opportunities as bars
        ax3.bar(periods, opportunities, alpha=0.3, color='green', label='Re-bidding Ops')
        
        ax3.set_xlabel('Time Period')
        ax3.set_ylabel('Distance / Opportunities', color='b')
        ax3_twin.set_ylabel('Utilization (%)', color='r')
        ax3.set_title('Dynamic Re-bidding Performance')
        ax3.tick_params(axis='y', labelcolor='b')
        ax3_twin.tick_params(axis='y', labelcolor='r')
        ax3.grid(True, alpha=0.3)
        
        # Combine legends
        lines1, labels1 = ax3.get_legend_handles_labels()
        lines2, labels2 = ax3_twin.get_legend_handles_labels()
        ax3.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
    # Plot 4: System efficiency metrics
    metrics_names = ['Total\nDistance', 'Avg\nUtilization', 'Assignment\nRate', 'Tasks\nAssigned']
    metrics_values = [
        final_metrics['total_distance'],
        final_metrics['avg_utilization'],
        final_metrics['assignment_rate'],
        final_metrics['total_tasks_assigned']
    ]
    
    # Normalize values for better visualization
    max_values = [max(metrics_values[0], metrics_values[1]), 100, 100, max(metrics_values[3])]
    normalized_values = [v / m * 100 for v, m in zip(metrics_values, max_values)]
    
    bars = ax4.bar(metrics_names, normalized_values, color=['red', 'blue', 'green', 'orange'], alpha=0.7)
    ax4.set_ylabel('Normalized Score')
    ax4.set_title('System Efficiency Metrics')
    ax4.grid(True, alpha=0.3)
    
    # Add actual values as text
    for bar, value, normalized in zip(bars, metrics_values, normalized_values):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 2,
                f'{value:.1f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Visualize results
visualize_multi_agent_results(agent_statuses, final_metrics, rebidding_history)

In [ ]:
# Emergent Behavior Analysis
def analyze_emergent_behavior(auctioneer, agent_statuses):
    """Analyze emergent behaviors in the multi-agent system"""
    print("=== EMERGENT BEHAVIOR ANALYSIS ===")
    
    # Analyze bidding patterns
    print("\n1. Bidding Pattern Analysis:")
    
    if auctioneer.auction_history:
        # Calculate bidding statistics
        bids_per_agent = defaultdict(list)
        winning_bids_per_agent = defaultdict(int)
        
        for auction in auctioneer.auction_history:
            winning_bids_per_agent[auction['winning_agent']] += 1
        
        print(f"   Total auctions: {len(auctioneer.auction_history)}")
        print(f"   Winning bids per agent:")
        for agent_id, wins in winning_bids_per_agent.items():
            print(f"     Agent {agent_id}: {wins} wins ({wins/len(auctioneer.auction_history)*100:.1f}%)")
    
    # Analyze workload distribution
    print("\n2. Workload Distribution:")
    
    workloads = [status['assigned_tasks'] for status in agent_statuses]
    utilizations = [status['utilization'] for status in agent_statuses]
    
    workload_variance = np.var(workloads)
    utilization_variance = np.var(utilizations)
    
    print(f"   Task assignment variance: {workload_variance:.2f}")
    print(f"   Utilization variance: {utilization_variance:.2f}")
    print(f"   Workload balance: {'Good' if workload_variance < 1 else 'Poor'}")
    
    # Analyze route efficiency
    print("\n3. Route Efficiency Analysis:")
    
    total_distance = final_metrics['total_distance']
    total_tasks = final_metrics['total_tasks_assigned']
    
    if total_tasks > 0:
        distance_per_task = total_distance / total_tasks
        print(f"   Distance per task: {distance_per_task:.2f}")
        print(f"   Total efficiency: {'High' if distance_per_task < 30 else 'Medium' if distance_per_task < 50 else 'Low'}")
    
    # Analyze agent specialization
    print("\n4. Agent Specialization Patterns:")
    
    for status in agent_statuses:
        agent_id = status['agent_id']
        tasks = status['assigned_tasks']
        utilization = status['utilization']
        
        if tasks == 0:
            specialization = "Idle"
        elif utilization > 80:
            specialization = "Highly Utilized"
        elif utilization > 50:
            specialization = "Moderately Utilized"
        else:
            specialization = "Underutilized"
        
        print(f"   Agent {agent_id}: {specialization} ({tasks} tasks, {utilization:.1f}% utilization)")
    
    # Analyze system-level properties
    print("\n5. System-Level Properties:")
    
    # Calculate Gini coefficient for workload inequality
    if len(workloads) > 1:
        sorted_workloads = sorted(workloads)
        n = len(sorted_workloads)
        cumulative_workloads = np.cumsum(sorted_workloads)
        gini = (n + 1 - 2 * sum(cumulative_workloads) / sum(sorted_workloads)) / n
        
        print(f"   Gini coefficient (workload inequality): {gini:.3f}")
        print(f"   Equality: {'High' if gini < 0.3 else 'Medium' if gini < 0.6 else 'Low'}")
    
    # Calculate system resilience
    active_agents = sum(1 for status in agent_statuses if status['assigned_tasks'] > 0)
    resilience = active_agents / len(agent_statuses)
    
    print(f"   System resilience: {resilience:.1%} ({active_agents}/{len(agent_statuses)} agents active)")
    
    return {
        'workload_variance': workload_variance,
        'utilization_variance': utilization_variance,
        'distance_per_task': distance_per_task if total_tasks > 0 else 0,
        'gini_coefficient': gini if len(workloads) > 1 else 0,
        'resilience': resilience
    }

# Analyze emergent behavior
emergent_properties = analyze_emergent_behavior(auctioneer, agent_statuses)

In [ ]:
# Compare Multi-Agent System with other methods
def compare_multi_agent_with_baselines(final_metrics, emergent_properties):
    """Compare Multi-Agent System with traditional approaches"""
    print("=== MULTI-AGENT VS TRADITIONAL METHODS ===")
    
    # Traditional method characteristics
    methods_comparison = {
        'Centralized Planning': {
            'coordination': 'Centralized',
            'adaptability': 'Low',
            'scalability': 'Limited',
            'robustness': 'Low',
            'complexity': 'High'
        },
        'Heuristic Methods': {
            'coordination': 'None',
            'adaptability': 'Low',
            'scalability': 'Good',
            'robustness': 'Medium',
            'complexity': 'Low'
        },
        'Metaheuristics': {
            'coordination': 'Centralized',
            'adaptability': 'Medium',
            'scalability': 'Medium',
            'robustness': 'Medium',
            'complexity': 'Medium'
        },
        'Machine Learning': {
            'coordination': 'Learned',
            'adaptability': 'High',
            'scalability': 'Good',
            'robustness': 'Medium',
            'complexity': 'High'
        },
        'Multi-Agent System': {
            'coordination': 'Decentralized',
            'adaptability': 'Very High',
            'scalability': 'Excellent',
            'robustness': 'High',
            'complexity': 'High'
        }
    }
    
    print(f"\nMethod Characteristics:")
    characteristics = ['coordination', 'adaptability', 'scalability', 'robustness', 'complexity']
    
    for char in characteristics:
        print(f"\n{char.title()}:")
        for method, props in methods_comparison.items():
            print(f"  {method:<20}: {props[char]}")
    
    print(f"\n=== MULTI-AGENT UNIQUE CAPABILITIES ===")
    
    print(f"\n1. Decentralized Decision Making:")
    print("   ✓ Autonomous agents make local decisions")
    print("   ✓ No single point of failure")
    print("   ✓ Distributed problem solving")
    print("   ✓ Local optimization with global awareness")
    
    print(f"\n2. Market-Based Coordination:")
    print(f"   ✓ Auction mechanism for task allocation")
    print(f"   ✓ Competitive bidding ensures efficiency")
    print(f"   ✓ Price signals coordinate resources")
    print(f"   ✓ Natural load balancing through competition")
    
    print(f"\n3. Emergent Intelligence:")
    if emergent_properties:
        print(f"   ✓ Gini coefficient: {emergent_properties['gini_coefficient']:.3f} (workload equality)")
        print(f"   ✓ System resilience: {emergent_properties['resilience']:.1%}")
        print(f"   ✓ Workload balance: {emergent_properties['workload_variance']:.2f} variance")
        print(f"   ✓ Route efficiency: {emergent_properties['distance_per_task']:.2f} per task")
    
    print(f"\n4. Dynamic Adaptation:")
    print("   ✓ Real-time re-bidding for optimization")
    print("   ✓ Task trading between agents")
    print("   ✓ Adaptive bidding strategies")
    print("   ✓ Self-organizing system behavior")
    
    print(f"\n5. Scalability Advantages:")
    print("   ✓ Linear scalability with number of agents")
    print("   ✓ Parallel processing of tasks")
    print("   ✓ No central bottleneck")
    print("   ✓ Heterogeneous agent capabilities")
    print("   ✓ Geographic distribution possible")
    
    print(f"\n=== WHEN TO USE MULTI-AGENT SYSTEMS ===")
    
    print(f"\nUse Multi-Agent Systems when:")
    print("• Distributed decision making is required")
    print("• System resilience is critical")
    print("• Real-time adaptation is essential")
    print("• Large-scale operations with many entities")
    print("• Complex stakeholder relationships exist")
    print("• Market-based coordination is appropriate")
    
    print(f"\nComplement with traditional methods when:")
    print("• Initial planning uses optimization algorithms")
    print("• Baseline solutions guide agent behavior")
    print("• Machine learning enhances agent capabilities")
    print("• Digital twins provide situational awareness")
    
    return methods_comparison

# Perform comparison
comparison_results = compare_multi_agent_with_baselines(final_metrics, emergent_properties)

### Why this Tier exists vs earlier Tiers
This Tier 6 provides decentralized autonomy that transforms traditional optimization:
- **Distributed Intelligence**: Agents make autonomous decisions locally
- **Market Coordination**: Auction mechanisms efficiently allocate resources
- **Emergent Behavior**: Complex patterns arise from simple agent rules
- **Dynamic Adaptation**: System self-optimizes through continuous re-bidding
- **Fault Tolerance**: No single point of failure in decision making

### Pros / Cons
**Pros:**
- Highly scalable to large numbers of agents and tasks
- Resilient to failures and disruptions
- Natural load balancing through competition
- Flexible and adaptable to changing conditions
- No central bottleneck in decision making

**Cons:**
- Complex to design and implement correctly
- May not guarantee global optimality
 - Requires sophisticated agent communication protocols
- Potential for strategic behavior between agents
- Higher computational overhead for coordination

### When to use this Tier
- **Large-scale operations** with many vehicles and customers
- **Distributed environments** where central control is impractical
- **Dynamic conditions** requiring rapid adaptation
- **Multi-stakeholder systems** with competing interests
- **High-reliability requirements** where fault tolerance is critical
- **Market-based logistics** where price signals coordinate resources

In [ ]:
# Final summary and key insights
print("=== MULTI-AGENT SYSTEM KEY INSIGHTS ===")
print()
print("1. System Performance:")
print(f"   Total distance: {final_metrics['total_distance']:.2f}")
print(f"   Average utilization: {final_metrics['avg_utilization']:.1f}%")
print(f"   Task assignment rate: {final_metrics['assignment_rate']:.1f}%")
print(f"   Active agents: {sum(1 for s in agent_statuses if s['assigned_tasks'] > 0)}/{len(agent_statuses)}")
print()

print("2. Agent Coordination:")
print(f"   Agents registered: {len(auctioneer.agents)}")
print(f"   Tasks created: {final_metrics['num_tasks']}")
print(f"   Auctions completed: {len(auctioneer.auction_history)}")
print(f"   Re-bidding rounds: {len(rebidding_history)}")
print(f"   Task trading events: {sum(h['rebidding_opportunities'] for h in rebidding_history)}")
print()

print("3. Emergent Properties:")
if emergent_properties:
    print(f"   Workload equality (Gini): {emergent_properties['gini_coefficient']:.3f}")
    print(f"   System resilience: {emergent_properties['resilience']:.1%}")
    print(f"   Route efficiency: {emergent_properties['distance_per_task']:.2f} km/task")
    print(f"   Workload balance variance: {emergent_properties['workload_variance']:.2f}")
print()

print("4. Technical Innovation:")
print("   ✓ Contract Net Protocol for agent communication")
print("   ✓ Market-based task allocation mechanism")
print("   ✓ Dynamic re-bidding for continuous optimization")
print("   ✓ Emergent self-organizing behavior")
print("   ✓ Fault-tolerant distributed architecture")
print()

print("5. Business Value:")
print("   ✓ Enhanced operational resilience")
print("   ✓ Automatic load balancing")
print("   ✓ Real-time adaptation to changes")
print("   ✓ Scalable to large operations")
print("   ✓ Reduced coordination overhead")
print()

print("6. System Characteristics:")
print("   - Decentralized decision making authority")
print("   - Market-based resource allocation")
   - Emergent intelligence from simple rules")
   - Self-organizing system behavior")
   - High fault tolerance and resilience")
print()

print("The Multi-Agent System demonstrates how")
print("decentralized intelligence can create robust,")
print("adaptive logistics systems that outperform")
print("traditional centralized approaches in complex,")
"dynamic environments.")